In [ ]:
import wandb
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Initialize wandb API
api = wandb.Api()

# Get all runs from project
runs = api.runs("your-username/barcode-detection")

# Extract metrics into dataframe
summary_list = []
config_list = []
name_list = []

for run in runs:
    # Get summary metrics
    summary_list.append(run.summary._json_dict)
    
    # Get config
    config_list.append({
        k: v for k, v in run.config.items() 
        if not k.startswith('_')
    })
    
    # Get run name
    name_list.append(run.name)

summary_df = pd.DataFrame(summary_list)
config_df = pd.DataFrame(config_list)
name_df = pd.DataFrame({'name': name_list})

# Combine into single dataframe
results_df = pd.concat([name_df, summary_df, config_df], axis=1)

# ============ VISUALIZATIONS ============

# 1. Compare mAP across experiments
plt.figure(figsize=(12, 6))
results_df = results_df.sort_values('test/mAP50-95', ascending=False)
plt.barh(results_df['name'], results_df['test/mAP50-95'])
plt.xlabel('mAP@0.5:0.95')
plt.title('Model Performance Comparison')
plt.tight_layout()
plt.savefig('experiment_comparison.png', dpi=300)

# 2. Augmentation impact analysis
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# Rotation impact
sns.scatterplot(data=results_df, x='augmentation.rotation_degrees', 
                y='test/mAP50-95', ax=axes[0,0], s=100)
axes[0,0].set_title('Impact of Rotation Augmentation')

# Blur impact
sns.scatterplot(data=results_df, x='augmentation.blur', 
                y='test/mAP50-95', ax=axes[0,1], s=100)
axes[0,1].set_title('Impact of Blur Augmentation')

# Mosaic impact
sns.scatterplot(data=results_df, x='augmentation.mosaic', 
                y='test/mAP50-95', ax=axes[1,0], s=100)
axes[1,0].set_title('Impact of Mosaic Augmentation')

# Image size impact
sns.scatterplot(data=results_df, x='img_size', 
                y='test/mAP50-95', hue='model_name', ax=axes[1,1], s=100)
axes[1,1].set_title('Impact of Image Resolution')

plt.tight_layout()
plt.savefig('augmentation_impact_analysis.png', dpi=300)

# 3. Create summary table
summary_table = results_df[[
    'name', 
    'test/mAP50', 
    'test/mAP50-95',
    'augmentation.rotation_degrees',
    'augmentation.blur',
    'augmentation.mosaic',
    'model_name',
    'img_size'
]].round(4)

print(summary_table.to_markdown())

# 4. Find best performing configuration
best_run = results_df.loc[results_df['test/mAP50-95'].idxmax()]
print("\n" + "="*60)
print("BEST PERFORMING CONFIGURATION:")
print("="*60)
print(f"Experiment: {best_run['name']}")
print(f"mAP@0.5:0.95: {best_run['test/mAP50-95']:.4f}")
print(f"Config: {best_run[config_df.columns].to_dict()}")